# Lab 1 at home · Two tables of your own in DuckDB

**Due:** Sunday, October 25 at 11:59 PM · **Submit:** your repo URL on CourseWorks

In class you loaded one table (penguins), found a problem and fixed it. Now do the same with **two related tables you choose**. In Session 2 we'll join them, so pick two that can be connected.

You will:
1. Pick two related public datasets and upload them
2. Load both into DuckDB and check what DuckDB decided about each column
3. Find the column the two tables share (the **key**) and check how well they match
4. Write five SQL queries that help explain your data
5. Write observations and a dataset card for each table

## 1. Pick two related tables

**Related** means they share a column you could match rows on: an ID, a name, a ZIP code, a country code, a date.

**Example pairs**
- **World Bank:** any indicator download (e.g. GDP per country per year) comes with a second file of country details (region, income group). The shared column is the country code.
- **Our World in Data:** two charts for the same countries, e.g. life expectancy and health spending. Shared columns: country and year.
- **NYC Open Data:** 311 complaints and population by ZIP code, or the street tree census and neighborhood boundaries. Shared column: ZIP code or neighborhood.
- **Your own idea:** any two public tables that share a column.

**Rules**
- **Public, open data only.** Your repo is public, and later this data goes to a free AI tier. Nothing from work, nothing personal, nothing behind a login.
- **CSV format.** If a file is Excel, open it and save it as CSV first.
- **Each file under 100 MB.** Between 1,000 and 1,000,000 rows is a good size for the main table; the second can be small.
- **Reuse allowed.** Check each license or terms page, and note it in your dataset card.

**Places to look:** [NYC Open Data](https://opendata.cityofnewyork.us) · [data.gov](https://data.gov) · [World Bank Open Data](https://data.worldbank.org) · [Our World in Data](https://ourworldindata.org) · [Kaggle Datasets](https://www.kaggle.com/datasets) (check each license)

## 2. Upload them

1. Download both CSVs to your computer.
2. In this Codespace, drag both files into the **`data`** folder in the file list on the left.
3. Rename each to something short with no spaces, e.g. `gdp.csv` and `countries.csv` (right-click → **Rename**).
4. Fill in the cell below and run it. Table names should be short, lowercase, no spaces: you'll type them in your SQL.

In [26]:
FILE_A  = "PLUTO.csv"     # <- your main table's file name
TABLE_A = "pluto"                   # <- a short name for it, e.g. "gdp"

FILE_B  = "POPS.csv"   # <- the related table's file name
TABLE_B = "pops"                 # <- a short name for it, e.g. "countries"

In [27]:
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_A = ROOT / "data" / FILE_A
DATA_B = ROOT / "data" / FILE_B

missing = [p.name for p in (DATA_A, DATA_B) if not p.exists()]
if missing:
    print("Can't find:", ", ".join(missing))
    print("Files in data/ right now:")
    for f in sorted((ROOT / "data").iterdir()):
        print("  ", f.name)
else:
    con = duckdb.connect(str(ROOT / "lab1_home.duckdb"))
    for p in (DATA_A, DATA_B):
        print(f"Found data/{p.name} ({p.stat().st_size / 1_000_000:.1f} MB)")
    print("Connected.")

Found data/PLUTO.csv (463.6 MB)
Found data/POPS.csv (0.2 MB)
Connected.


## 3. First look

Preview the first 10 rows of each raw file before loading.

In [28]:
con.sql(f"SELECT * FROM read_csv('{DATA_A}') LIMIT 10")

┌─────────┬───────────┬─────────┬─────────────────┬───────────────────┬────────┬────────────┬──────────────────┬──────────┬──────────┬────────────┬────────────┬───────────┬──────────┬───────────────────────────┬───────────┬───────────┬───────────┬───────────┬──────────┬──────────┬─────────┬─────────┬─────────┬───────────┬───────────┬───────────┬─────────┬───────────┬───────────┬──────────────────────────────┬─────────┬──────────┬─────────┬─────────┬────────────┬────────────┬────────────┬───────────┬────────────┬───────────┬────────────┬──────────┬───────────┬──────────┬────────────┬──────────┬──────────┬───────────┬───────────┬─────────┬──────────┬────────────┬─────────┬──────────┬────────────┬───────────┬───────────┬───────────┬────────────┬────────────┬──────────┬──────────┬──────────┬──────────┬─────────┬──────────┬───────────┬────────┬──────────┬────────────┬─────────┬───────────┬─────────┬────────┬────────────┬─────────────┬─────────┬─────────┬─────────┬────────┬───────────┬──────

In [29]:
con.sql(f"SELECT * FROM read_csv('{DATA_B}') LIMIT 10")

┌─────────────┬───────────┬──────────┬─────────────────┬─────────┬────────────────┬──────────┬────────────────────────────────────────┬───────────┬─────────┬──────────────────────┬────────────────────────────────────────────────────────────────────────────────┬────────────────┬──────────────────────┬──────────────────────────────────────────┬───────────────────────────────────────────────────┬────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────┬─────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────┬─────────────────────┬───────────┬────────────┬──

## 4. Load both and check the types

Load each file into a table, then `DESCRIBE` it. Compare each column's type with what it *should* be. A number or date column stored as `VARCHAR` (text) is the same kind of problem you fixed in class.

In [30]:
for table, path in ((TABLE_A, DATA_A), (TABLE_B, DATA_B)):
    con.sql(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM read_csv('{path}', thousands=',')")
    rows = con.sql(f"SELECT count(*) FROM {table}").fetchone()[0]
    print(f"{table}: {rows:,} rows loaded")

pluto: 858,284 rows loaded
pops: 392 rows loaded


In [31]:
con.sql(f"DESCRIBE {TABLE_A}")

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ borough           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Tax block         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Tax lot           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ community board   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ census tract 2010 │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ cb2010            │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ schooldist        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ council district  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ postcode          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ firecomp  

In [32]:
con.sql(f"DESCRIBE {TABLE_B}")

┌────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name           │ column_type │  null   │   key   │ default │  extra  │
│            varchar             │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ POPS_Number                    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Borough                        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Borocode                       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Community Board                │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Number                         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Street                         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Postcode                       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Building_Address_Wi

`SUMMARIZE` profiles every column at once: min, max, how many distinct values, and `null_percentage` (how much is missing).

In [33]:
con.sql(f"SUMMARIZE {TABLE_A}")

┌───────────────────┬─────────────┬─────────────────────────────────┬───────────────────────────────────┬───────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬───────────────────┬────────┬─────────────────┐
│    column_name    │ column_type │               min               │                max                │ approx_unique │        avg         │        std         │        q25         │        q50        │        q75        │ count  │ null_percentage │
│      varchar      │   varchar   │             varchar             │              varchar              │     int64     │      varchar       │      varchar       │      varchar       │      varchar      │      varchar      │ int64  │  decimal(9,2)   │
├───────────────────┼─────────────┼─────────────────────────────────┼───────────────────────────────────┼───────────────┼────────────────────┼────────────────────┼────────────────────┼───────────────────┼───────────────────┼────────┼───────────

In [34]:
con.sql(f"SUMMARIZE {TABLE_B}")

┌────────────────────────────────┬─────────────┬──────────────────────────────────────────┬────────────────────────────────────┬───────────────┬────────────────────┬──────────────────────┬──────────────┬────────────────────┬────────────────────┬───────┬─────────────────┐
│          column_name           │ column_type │                   min                    │                max                 │ approx_unique │        avg         │         std          │     q25      │        q50         │        q75         │ count │ null_percentage │
│            varchar             │   varchar   │                 varchar                  │              varchar               │     int64     │      varchar       │       varchar        │   varchar    │      varchar       │      varchar       │ int64 │  decimal(9,2)   │
├────────────────────────────────┼─────────────┼──────────────────────────────────────────┼────────────────────────────────────┼───────────────┼────────────────────┼───────────────────

In [35]:
summarya = con.sql(f"SUMMARIZE {TABLE_A}").df()
summaryb = con.sql(f"SUMMARIZE {TABLE_B}").df()

print("TABLE A")
display(summarya[summarya["column_name"].str.contains("BBL", case=False, na=False)])

print("TABLE B")
display(summaryb[summaryb["column_name"].str.contains("BBL", case=False, na=False)])

TABLE A


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
70,BBL,BIGINT,1000010010,5080500094,950940,3521236866.3682313,1031516390.5705348,3031141135,4009065611,4107094566,858284,0.00
82,appbbl,BIGINT,1000010010,5080490030,44611,3728993670.9334097,1221050880.6720846,3022521730,4044544304,5019310247,858284,88.28


TABLE B


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
31,BBL,BIGINT,1000047501,5019550001,406,1160060751.9871795,594181128.9000524,1008883544,1012711081,1014093623,392,0.51


## 5. Fix what loaded wrong (if anything)

If every type looks right, write "No fixes needed" in the text cell after the next one and skip ahead.

Otherwise, open the CSV (double-click it in the `data` folder) and find **how missing values are written** in the columns that came in as text. Edit the `nullstr` list below to match, and run it for the table that needs fixing.

In [36]:
con.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_A} AS
    SELECT * FROM read_csv(
        '{DATA_A}',
        nullstr=['NA', 'N/A', ''],
        thousands=','
    )
""")

con.sql(f"DESCRIBE {TABLE_A}").df()

,column_name,column_type,null,key,default,extra
0,borough,VARCHAR,YES,None,None,None
1,Tax block,BIGINT,YES,None,None,None
2,Tax lot,BIGINT,YES,None,None,None
3,community board,BIGINT,YES,None,None,None
4,census tract 2010,DOUBLE,YES,None,None,None
...,...,...,...,...,...,...
103,landmkdate,VARCHAR,YES,None,None,None
104,masdate,VARCHAR,YES,None,None,None
105,polidate,VARCHAR,YES,None,None,None
106,rpaddate,VARCHAR,YES,None,None,None


**What did you fix, and how did you know?** (Double-click to edit.)



## 6. Find the key: how do the two tables connect?

The **key** is the column both tables share. Type its name in each table below. It may be spelled differently in each, e.g. `Country Code` in one and `iso3` in the other.

The next cell counts the distinct key values in each table and how many of them appear in both. (It treats both as text, so a ZIP code stored as a number still matches one stored as text.)

In [37]:
KEY_A = "BBL"   # <- the shared column's name in your main table
KEY_B = "BBL"   # <- its name in the second table

In [38]:
con.sql(f"""
    WITH a AS (SELECT DISTINCT CAST("{KEY_A}" AS VARCHAR) AS k FROM {TABLE_A}),
         b AS (SELECT DISTINCT CAST("{KEY_B}" AS VARCHAR) AS k FROM {TABLE_B})
    SELECT
        (SELECT count(*) FROM a)                           AS keys_in_a,
        (SELECT count(*) FROM b)                           AS keys_in_b,
        (SELECT count(*) FROM a WHERE k IN (SELECT k FROM b)) AS keys_in_both
""")

┌───────────┬───────────┬──────────────┐
│ keys_in_a │ keys_in_b │ keys_in_both │
│   int64   │   int64   │    int64     │
├───────────┼───────────┼──────────────┤
│    858284 │       391 │          385 │
└───────────┴───────────┴──────────────┘

**Describe the relationship** (double-click to edit):
--"No fixes needed"
- **Key column:** in the main table it's called BBL, in the second BBL.
- **How they relate:** does one row in the second table match *one* row in the main table, or *many*? (e.g. one country has many GDP rows, one per year)
-- Each row is connected to one row.
- **How well they match:** how many keys appear in only one table? Give one example and a guess at why (spelling, codes vs. names, regions mixed in with countries...)
-- PLUTO has every single tax lots data and BBL in the city, but POPS only includes BBL of tax lots with POPS. 



## 7. Five queries that explain your data

Write one query of each kind. Under each, double-click the text cell and write one line: **what does this result tell us?** Use either table.

Tip: if a column name has spaces or capitals, wrap it in double quotes: `"Country Name"`. Start from a query in the in-class notebook and change one thing.

### Query 1 · Size and shape
*How many rows, and over what time period or area?* Useful SQL: `count(*), min(), max()`

In [39]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT borough) AS borough_count
    FROM {TABLE_B}
""").df()

,total_rows,borough_count
0,392,5


**What this tells us:** There are 392 POPS in across all 5 borough of New York.

### Query 2 · Filter
*Which rows meet a condition you care about?* Useful SQL: `WHERE`

In [40]:
con.sql(f"""
    SELECT COUNT(*) AS manhattan_pops
    FROM {TABLE_B
          }
    WHERE Borocode = 1
    LIMIT 10
""").df()

,manhattan_pops
0,366


**What this tells us:** 366 POPS out of 392 are in Manhattan, indicating most of the pops are concentrated in this borough.

### Query 3 · Group
*How does a count or average differ across categories?* Useful SQL: `GROUP BY`

In [41]:
con.sql(f"""
    SELECT
        Borocode,
        COUNT(*) AS pops_count
    FROM {TABLE_B}
    GROUP BY Borocode
    ORDER BY pops_count DESC
""").df()

,Borocode,pops_count
0,1,366
1,3,16
2,4,5
3,2,3
4,5,2


**What this tells us:** POPS are heavily concentrated in Manhattan (366 records), while the other four boroughs have significantly fewer, with Brooklyn having 16, Queens 5, the Bronx 3, and Staten Island 2.

### Query 4 · Rank
*What are the top or bottom 10 of something?* Useful SQL: `ORDER BY ... LIMIT 10`

In [42]:
con.sql(f"""
    SELECT
        Public_Space_Type,
        COUNT(*) AS pops_count
    FROM {TABLE_B}
    WHERE Public_Space_Type IS NOT NULL
    GROUP BY Public_Space_Type
    ORDER BY pops_count DESC
    LIMIT 10
""").df()

,Public_Space_Type,pops_count
0,Plaza,90
1,Residential Plaza,56
2,Arcade; Plaza,55
3,Urban Plaza,14
4,Public Plaza,12
5,Public Access Area,12
6,Through Block Connection,7
7,Arcade,6
8,Covered Pedestrian Space,5
9,Arcade; Sidewalk Widening; Urban Plaza,5


**What this tells us:** Plaza is the most common type of POPS, despite there being several different types of plaza.

### Query 5 · Quality check
*What's missing, duplicated or impossible?* Useful SQL: `count(*) - count(column), count(DISTINCT ...)`

In [43]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(BBL) AS missing_bbl,
        COUNT(BBL) - COUNT(DISTINCT BBL) AS duplicate_bbl
    FROM {TABLE_B}
""").df()

,total_rows,missing_bbl,duplicate_bbl
0,392,2,0


**What this tells us:** There are only 2 missing rows in BBL.

## 8. Three observations

Specific things you noticed that you'd want to check before trusting this data. Point to numbers.

1. 366 out of 392 POPS records (93.4%) are located in Manhattan. This large geographic imbalance should be considered when comparing boroughs.
2. 2 out of 392 POPS records have missing BBL values, which could prevent them from being matched with PLUTO.
3. Of the 390 POPS records with valid BBL values, 385 match PLUTO, while 5 do not. These unmatched records need further investigation to determine whether the datasets use different or outdated parcel identifiers.

**One question you'd want a non-technical person to be able to ask of this data:**

How many privately owned public spaces are in my neighborhood, and what types of spaces are they?

## 9. Dataset cards

| | Main table | Second table |
|---|---|---|
| **Dataset name** | Primary Land Use Tax Lot Output (PLUTO) | Privately Owned Public Spaces (POPS) |
| **Source (link)** | https://data.cityofnewyork.us/City-Government/Primary-Land-Use-Tax-Lot-Output-PLUTO-/64uk-42ks/about_data | https://data.cityofnewyork.us/City-Government/Privately-Owned-Public-Spaces-POPS-/rvih-nhyn/about_data |
| **Who collected it, and why** | NYC Department of City Planning (DCP); to document tax lot, land use, building and zoning characteristics | NYC DCP, originally developed with Jerold S. Kayden and Municipal Art Society; to document privately owned public spaces |
| **License or terms of use** | NYC Open Data Terms of Use | NYC Open Data Terms of Use |
| **Rows × columns** | 858,000 × 108 | 392 × 34 |
| **Time period covered** | 2026 snapshot (version 26v2) | Historical POPS records; current published version 25v2 (2025) |
| **How often it's updated** | Major updates quarterly; minor zoning updates monthly | Scheduled every 6 months |

## 10. Submit

1. Run the cell below to close the connection.
2. Fill in the lifecycle map of your data in `labs/lab1.md`.
3. **Save this notebook (Cmd+S)**, then commit and push: **Source Control** → message `Lab 1` → **Commit** → **Sync Changes**.
4. Check on GitHub that both CSVs, both notebooks and `labs/lab1.md` are there, with results visible.
5. Paste your repo URL into the Lab 1 assignment on CourseWorks.

In [46]:
con.close()
print("Done. Save, commit, push, and submit your repo URL.")

Done. Save, commit, push, and submit your repo URL.
